<a href="https://colab.research.google.com/github/zia207/High_Performance_Computing_R/blob/main/Notebook/02_02_02_hpc_parallel_processing_future_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=1LhRCsnq8s3y0M76oN3-GEUfWQVL_yUv-)


# Data Processing and Analysis with {future}


Efficient data processing and analysis are critical in modern data science, especially as datasets grow in size and complexity. In R, the {future} package offers a powerful and intuitive framework for parallel and distributed computing, enabling users to write scalable and efficient code with minimal changes to their existing workflows. By abstracting the complexities of parallel backends—such as multi-core processing, distributed computing, or even cloud-based resources —{future} allows analysts and researchers to focus on their analytical tasks while significantly reducing computation time. This introduction explores how {future} integrates seamlessly with common R programming patterns, enhancing performance in data manipulation, modeling, and simulation through easy-to-implement parallelization strategies. Whether you're working with large datasets or running computationally intensive analyses, {future} empowers R users to harness the full potential of modern computing architectures.

The {future} package provides a more flexible and user-friendly framework for parallel and asynchronous computation in R, allowing seamless switching between sequential and parallel evaluation with minimal code changes. Below, we show how to use {future} for descriptive statistics, correlation, simple linear regression, and multiple linear regression, leveraging the `future_lapply()` function from the {future.apply} package for parallel processing.

![future logo](https://future.futureverse.org/logo.png)


## Use Cases

- Speeding up bootstrapping or Monte Carlo simulations.
- Parallel model training across hyperparameters.
- Processing large lists or data chunks.
- Accelerating Shiny apps with background computation.
- Scaling data pipelines in `targets` or `drake`.


## Setup R in Python Runtime — Install {rpy2}

{rpy2} allows running R code in Colab's Python runtime via `%%R` magic.


In [ ]:
!pip uninstall rpy2 -y
!pip install rpy2==3.5.1
%load_ext rpy2.ipython


## Mount Google Drive

Mount Google Drive if your data or R library folder is stored there. Adjust `dataFolder` in the data-setup cell to point at your Parquet file (for example under `MyDrive`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Check and Install Required R Packages

The cells below mirror the Quarto tutorial: define packages, install to Drive, verify, load.


### Define required packages


In [ ]:
%%R
packages <- c(
  'tidyverse',
  'data.table',
  'feather',
  'arrow',
  'parallel',
  'foreach',
  'doParallel',
  'future',
  'furrr',
  'future.apply',
  'parallelly',
  'nycflights13',
  'glue'
)


### Install missing packages (Google Drive library)


In [ ]:
%%R
new.packages <- packages[!(packages %in% installed.packages(lib = "drive/My Drive/R/")[, "Package"])]
if (length(new.packages)) install.packages(new.packages, lib = "drive/My Drive/R/")


### Verify installation


In [ ]:
%%R
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))


### Load packages


In [ ]:
%%R
.libPaths(c("drive/My Drive/R/", .libPaths()))
loaded_packages <- lapply(packages, function(pkg) {
  if (requireNamespace(pkg, quietly = TRUE)) {
    suppressPackageStartupMessages(library(pkg, character.only = TRUE))
    TRUE
  } else {
    FALSE
  }
})
names(loaded_packages) <- packages


### Check loaded packages


In [ ]:
%%R
cat("Successfully loaded packages:\n")
print(search()[grepl("package:", search())])


## Overview of {future} Package

The `{future}` package in R is a powerful and flexible framework designed to simplify parallel and asynchronous computing, enabling users to write scalable, efficient, and portable code. It provides a unified interface for evaluating R expressions in the "future" — that is, asynchronously or in parallel — without requiring deep expertise in low-level parallel programming. This makes it an ideal tool for speeding up data processing, simulations, modeling, and other computationally intensive tasks.

### Core Concept: The Future

At the heart of the package is the **future** abstraction — a placeholder for a value that will be computed later. Once created, a future can be resolved (i.e., its value retrieved) when needed. The computation may happen:

- In the background on the same R session (synchronous or asynchronous),
- On a different R session (e.g., a forked process),
- On a different core (multicore),
- On a different machine (cluster or cloud).

Here is an example illustrating how the basics of futures work. First, consider the following code snippet that uses plain R code:


In [ ]:
%%R
v <- {
  cat("Hello world!\n")
  3.14
}
v


It works by assigning the value of an expression to variable v and we then print the value of v. Moreover, when the expression for v is evaluated we also print a message to the console. Now, let's see how we can achieve the same using a future:


In [ ]:
%%R
plan(sequential)
v %<-% {
  cat("Hello world!\n")
  3.14
}
v


The difference is that the expression is now wrapped in a future assignment operator `%<-%`. This means that the expression will be evaluated in the {future}, potentially in a different R session or core (`<-`), depending on the plan we set. The message "Hello world!" will be printed when the future is resolved (i.e., when we access the value of v).

We can choose to evaluate the future expression in a separate R process asynchronously by simply switching settings as:


In [ ]:
%%R
plan(multisession)  # Use multiple R sessions
v %<-% {
  cat("Hello world!\n")
  3.14
}
v


### Key Features

#### Unified API

The `future` package provides a consistent syntax regardless of the backend used. You write your code once and switch between sequential, multicore, multisession, or cluster execution by changing the **plan**.

#### Multiple Backends (Execution Strategies)

You can control how futures are evaluated using `plan()`:

| Plan           | Description                                        |
|----------------|----------------------------------------------------|
| `sequential`   | No parallelism (default)                           |
| `multicore`    | Forked processes (Unix-like systems)               |
| `multisession` | Separate R sessions (Windows & Unix)               |
| `multiprocess` | Auto-selects multicore or multisession             |
| `cluster`      | Custom clusters (e.g., across machines)            |
| `remote`       | Remote R sessions                                  |
| `batchtools`   | Integration with job schedulers (SLURM, SGE, etc.) |
| `future.callr` | Uses `{callr}` for lightweight parallelism         |


In [ ]:
%%R
plan(multisession)  # Automatically chooses best option


In [ ]:
%%R
# A sequential future
plan(sequential)
f <- future({
  a <- 7
  b <- 3
  c <- 2
  a * b * c
})
y <- value(f)
print(y)


In [ ]:
%%R
# A sequential future with lazy evaluation
plan(sequential)
f <- future({
  a <- 7
  b <- 3
  c <- 2
  a * b * c
}, lazy = TRUE)
y <- value(f)
print(y)


**Note (Colab / `rpy2`):** `plan("multicore")` uses forked workers and may not work in all hosted environments. If a cell errors, switch to `plan(multisession)` for separate R processes.


In [ ]:
%%R
# A multicore future (specified as a string) — use multisession on Colab if this fails
tryCatch({
  plan("multicore")
  f <- future({
    a <- 7
    b <- 3
    c <- 2
    a * b * c
  })
  y <- value(f)
  print(y)
}, error = function(e) {
  message("multicore failed: ", e$message, " — trying multisession")
  plan(multisession)
  f <- future({
    a <- 7
    b <- 3
    c <- 2
    a * b * c
  })
  print(value(f))
})


In [ ]:
%%R
# A multisession future (specified via a string variable)
plan("future::multisession")
f <- future({
  a <- 7
  b <- 3
  c <- 2
  a * b * c
})
y <- value(f)
print(y)


In [ ]:
%%R
## Explicitly specifying number of workers
## (default is parallelly::availableCores())
plan(multisession, workers = min(10L, max(1L, availableCores() - 1L)))
message("Number of parallel workers: ", nbrOfWorkers())


### Integration with Other Packages

The `future` ecosystem integrates seamlessly with popular R packages through **future-aware** counterparts:

- **`furrr`**: Parallel `purrr` – apply `map()` functions in parallel.


In [ ]:
%%R
library(furrr)
plan(multisession)
mtcars %>%
  split(.$cyl) %>%  # Split into list by 'cyl'
  future_map_dbl(~ coef(lm(mpg ~ wt, data = .x))[2])  # Extract slope


- **`future.apply`**: Parallel versions of `lapply`, `sapply`, etc.


In [ ]:
%%R
library(future.apply)
plan(multisession)

plan(sequential)  # No parallelism — safe
result <- future_lapply(1:10, function(x) x^2)
print(result)


- **`future.batchtools`**: For HPC environments (e.g., SLURM, PBS).
- **`clustermq`**: Efficient worker-based parallelization.
- **`targets` / `drake`**: Scalable pipelines with future backends.


### Handling Globals and Dependencies

The `future` package automatically identifies global variables and functions used inside a future expression and exports them to the worker environment. You can control this behavior:

- Use `globals = FALSE` to disable automatic detection.
- Specify required objects with `globals = list(...)`.
- Use `local = TRUE` to avoid cluttering global environment.


In [ ]:
%%R
a <- 5
f <- future(a^2, globals = "a")
value(f)


### Error Handling and Debugging

If a future throws an error, it is captured and re-thrown when `value(f)` is called:


In [ ]:
%%R
f <- future(stop("Something went wrong"))
tryCatch({
  value(f)
}, error = function(e) print(paste("Error:", e$message)))


For debugging, use `plan(sequential)` to run futures in the current session.


### Nested Futures and Dynamic Scheduling

With `plan(list(...))`, you can combine multiple backends or support **nested parallelism** (e.g., foreach inside a future). The `future.callr::callr` and `future.devices` packages further extend capabilities.


In [ ]:
%%R
plan(list(tweak(multisession, workers = 2),
          tweak(multisession, workers = min(4L, max(1L, availableCores() - 1L)))))


### Best Practices

- Use `plan(multiprocess)` for general-purpose parallelism.
- Always test with `plan(sequential)` first.
- Avoid side effects (e.g., `write.csv()` inside a future).
- Be mindful of memory usage — each worker may copy data.
- Use `furrr` or `future.apply` for functional programming patterns.


## Parallel Data Processing with {future}

In this tutorial, you'll learn how to use the {future} package in R to efficiently process large datasets in parallel. We'll work with the Yellow Taxi Trip Data for 2023, a real-world dataset stored in Parquet format (`yellow_tripdata_2023.parquet`), which is commonly used in data analysis and can be quite large (often several GBs).

We'll demonstrate how {future} can speed up data processing tasks such as filtering, aggregation, and summary statistics by leveraging multiple CPU cores.

### Data

Download full-year Yellow Taxi Parquet from the [NYC TLC trip record data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) if needed, or use a subset. Set `dataFolder` below.


### Set `dataFolder` for Colab (edit path)


In [ ]:
%%R
# Colab: set folder containing yellow_tripdata_2023.parquet
dataFolder <- file.path("/content/drive/MyDrive", "YourFolder", "Data")
dataFolder <- sub("/$", "", dataFolder)
dataFolder <- paste0(dataFolder, "/")


In [ ]:
%%R
# Local / repo path (edit if running outside this project)
dataFolder <- "/home/zia207/WebSites/R_Website/High_Performance_Computing_R/R_Markdown/Data/"
taxi_data <- read_parquet(paste0(dataFolder, "yellow_tripdata_2023.parquet"))


### Set Up Parallel Backend with {future}

Choose a parallelization strategy. We'll use `multisession` for broad compatibility in Colab; on a local Linux session you can use `plan(multicore)` or `plan(multiprocess)`.


In [ ]:
%%R
plan(multisession, workers = min(max(1L, availableCores() - 1L), 8L))
plan()


### Splitting the Data

To parallelize, we split the dataset into chunks based on the number of cores (here, by month):


In [ ]:
%%R
taxi_by_month <- taxi_data %>%
  mutate(month_val = month(tpep_pickup_datetime)) %>%
  group_by(month_val) %>%
  group_split() %>%
  map(collect)

length(taxi_by_month)  # Should be 12


### Parallel Processing with `future_map()`


In [ ]:
%%R
library(furrr)
plan(multisession, workers = min(max(1L, availableCores() - 1L), 8L))
monthly_summary <- future_map_dfr(taxi_by_month, ~ {
  month_num <- first(.x$month_val)

  .x %>%
    summarise(
      month = month_num,
      total_trips = n(),
      avg_fare = mean(fare_amount, na.rm = TRUE),
      total_revenue = sum(total_amount, na.rm = TRUE)
    )
})

print(monthly_summary)


### Visualizing Results


In [ ]:
%%R
monthly_summary %>%
  arrange(month) %>%
  mutate(month_name = month.abb[month]) %>%
  ggplot(aes(x = month_name, y = total_revenue)) +
  geom_col(fill = "steelblue") +
  labs(title = "Total Revenue by Month (Yellow Taxi 2023)",
       x = "Month", y = "Total Revenue ($)") +
  theme_minimal()


### Parallel Filtering and Saving

Suppose we want to save each month's data as a CSV (e.g., for downstream use):


In [ ]:
%%R
# Create output directory (adjust path as needed)
dir.create("output/monthly_csv", showWarnings = FALSE, recursive = TRUE)

future_map2(taxi_by_month, seq_along(taxi_by_month), ~ {
  filename <- glue("output/monthly_csv/yellow_tripdata_2023_{.y}.csv")
  readr::write_csv(.x, filename)
  message(glue("Saved {.y}: {nrow(.x)} rows"))
})


## Basic Statistical Analysis of NYC Taxi Data with {future}

The {future} package provides a unified framework for parallel and distributed processing in R, making it easy to parallelize computations with minimal code changes. This tutorial adapts the statistical analysis of the NYC Yellow Taxi Trip Data to use {future} and {future.apply}. We will perform:

- **Descriptive Statistics**: Mean, median, standard deviation, min, and max for trip_duration, fare_amount, and trip_distance.
- **Correlation Analysis**: Pearson correlation matrix for trip_duration, fare_amount, and trip_distance.
- **Simple Linear Regression**: Predict fare_amount from trip_duration.
- **Multiple Linear Regression**: Predict fare_amount from trip_duration, trip_distance, and passenger_count.

The {future} package allows us to switch between sequential and parallel evaluation by setting the evaluation plan, making it ideal for scaling analyses on large datasets.


### Setting Up the Future Backend

Load the `{future}` and `{future.apply}` packages and set the evaluation plan. We use `multisession` for cross-platform compatibility.


In [ ]:
%%R
plan(multisession, workers = max(1L, availableCores() - 1L))
print(availableCores())


The `workers` argument specifies the number of parallel processes. We use `availableCores() - 1` to reserve one core for system tasks.


### Loading and Preparing the Data

Load the Parquet file using `{arrow}` and preprocess the data (reuse `taxi_data` from above, or load again if you skipped the parallel section):


In [ ]:
%%R
taxi_data <- read_parquet(paste0(dataFolder, "yellow_tripdata_2023.parquet")) %>%
  mutate(trip_duration = as.numeric(difftime(tpep_dropoff_datetime, tpep_pickup_datetime, units = "mins"))) %>%
  filter(
    !is.na(trip_duration), !is.na(fare_amount), !is.na(trip_distance), !is.na(passenger_count),
    trip_duration > 0, fare_amount > 0, trip_distance > 0, passenger_count > 0
  ) %>%
  select(trip_duration, fare_amount, trip_distance, passenger_count)

glimpse(taxi_data)


### Splitting the Data


In [ ]:
%%R
n_chunks <- max(1L, availableCores() - 1L)
data_splits <- split(taxi_data, rep(1:n_chunks, each = ceiling(nrow(taxi_data) / n_chunks), length.out = nrow(taxi_data)))


### Descriptive Statistics

Compute mean, median, standard deviation, min, and max for `trip_duration`, `fare_amount`, and `trip_distance`.

#### Define the Descriptive Statistics Function


In [ ]:
%%R
process_descriptive <- function(chunk) {
  chunk %>%
    summarise(
      mean_duration = mean(trip_duration, na.rm = TRUE),
      median_duration = median(trip_duration, na.rm = TRUE),
      sd_duration = sd(trip_duration, na.rm = TRUE),
      min_duration = min(trip_duration, na.rm = TRUE),
      max_duration = max(trip_duration, na.rm = TRUE),
      mean_fare = mean(fare_amount, na.rm = TRUE),
      median_fare = median(fare_amount, na.rm = TRUE),
      sd_fare = sd(fare_amount, na.rm = TRUE),
      min_fare = min(fare_amount, na.rm = TRUE),
      max_fare = max(fare_amount, na.rm = TRUE),
      mean_distance = mean(trip_distance, na.rm = TRUE),
      median_distance = median(trip_distance, na.rm = TRUE),
      sd_distance = sd(trip_distance, na.rm = TRUE),
      min_distance = min(trip_distance, na.rm = TRUE),
      max_distance = max(trip_distance, na.rm = TRUE)
    ) %>%
    collect()
}


#### Serial Descriptive Statistics (Baseline)


In [ ]:
%%R
plan(sequential)
system.time({
  serial_desc <- lapply(data_splits, process_descriptive)
  serial_desc_summary <- do.call(rbind, serial_desc) %>%
    summarise(across(everything(), mean, na.rm = TRUE))
})

print(serial_desc_summary)


#### Parallel Descriptive Statistics with `future_lapply`


In [ ]:
%%R
plan(multisession, workers = max(1L, availableCores() - 1L))
system.time({
  parallel_desc <- future_lapply(data_splits, process_descriptive)
  parallel_desc_summary <- do.call(rbind, parallel_desc) %>%
    summarise(across(everything(), mean, na.rm = TRUE))
})

print(parallel_desc_summary)


The `future_lapply()` function is a drop-in replacement for `lapply()`, automatically handling parallel evaluation based on the `plan()` setting.

### Correlation Analysis

Compute the Pearson correlation matrix for `trip_duration`, `fare_amount`, and `trip_distance`.

#### Define the Correlation Function


In [ ]:
%%R
process_correlation <- function(chunk) {
  cor_matrix <- cor(chunk[, c("trip_duration", "fare_amount", "trip_distance")], method = "pearson", use = "complete.obs")
  as.vector(cor_matrix)
}


#### Serial Correlation


In [ ]:
%%R
plan(sequential)
system.time({
  serial_cor <- lapply(data_splits, process_correlation)
  serial_cor_mean <- colMeans(do.call(rbind, serial_cor), na.rm = TRUE)
  cor_matrix <- matrix(serial_cor_mean, nrow = 3, dimnames = list(
    c("trip_duration", "fare_amount", "trip_distance"),
    c("trip_duration", "fare_amount", "trip_distance")
  ))
})

print(round(cor_matrix, 3))


#### Parallel Correlation with `future_lapply`


In [ ]:
%%R
plan(multisession, workers = max(1L, availableCores() - 1L))
system.time({
  parallel_cor <- future_lapply(data_splits, process_correlation)
  parallel_cor_mean <- colMeans(do.call(rbind, parallel_cor), na.rm = TRUE)
  parallel_cor_matrix <- matrix(parallel_cor_mean, nrow = 3, dimnames = list(
    c("trip_duration", "fare_amount", "trip_distance"),
    c("trip_duration", "fare_amount", "trip_distance")
  ))
})

print(round(parallel_cor_matrix, 3))


### Simple Linear Regression

Model `fare_amount` as a function of `trip_duration`.

#### Define the Simple Regression Function


In [ ]:
%%R
process_simple_regression <- function(chunk) {
  model <- lm(fare_amount ~ trip_duration, data = chunk)
  coef <- summary(model)$coefficients
  list(
    intercept = coef[1, 1],
    slope = coef[2, 1],
    r_squared = summary(model)$r.squared
  )
}


#### Serial Simple Regression


In [ ]:
%%R
plan(sequential)
system.time({
  serial_reg <- lapply(data_splits, process_simple_regression)
  serial_reg_summary <- do.call(rbind, lapply(serial_reg, as.data.frame)) %>%
    summarise(
      avg_intercept = mean(intercept, na.rm = TRUE),
      avg_slope = mean(slope, na.rm = TRUE),
      avg_r_squared = mean(r_squared, na.rm = TRUE)
    )
})

print(serial_reg_summary)


#### Parallel Simple Regression with `future_lapply`


In [ ]:
%%R
plan(multisession, workers = max(1L, availableCores() - 1L))
system.time({
  parallel_reg <- future_lapply(data_splits, process_simple_regression)
  parallel_reg_summary <- do.call(rbind, lapply(parallel_reg, as.data.frame)) %>%
    summarise(
      avg_intercept = mean(intercept, na.rm = TRUE),
      avg_slope = mean(slope, na.rm = TRUE),
      avg_r_squared = mean(r_squared, na.rm = TRUE)
    )
})

print(parallel_reg_summary)


### Multiple Linear Regression

Model `fare_amount` as a function of `trip_duration`, `trip_distance`, and `passenger_count`.

#### Define the Multiple Regression Function


In [ ]:
%%R
process_multiple_regression <- function(chunk) {
  model <- lm(fare_amount ~ trip_duration + trip_distance + passenger_count, data = chunk)
  coef <- summary(model)$coefficients
  list(
    intercept = coef[1, 1],
    slope_duration = coef[2, 1],
    slope_distance = coef[3, 1],
    slope_passenger = coef[4, 1],
    r_squared = summary(model)$r.squared
  )
}


#### Serial Multiple Regression


In [ ]:
%%R
plan(sequential)
system.time({
  serial_multi_reg <- lapply(data_splits, process_multiple_regression)
  serial_multi_reg_summary <- do.call(rbind, lapply(serial_multi_reg, as.data.frame)) %>%
    summarise(
      avg_intercept = mean(intercept, na.rm = TRUE),
      avg_slope_duration = mean(slope_duration, na.rm = TRUE),
      avg_slope_distance = mean(slope_distance, na.rm = TRUE),
      avg_slope_passenger = mean(slope_passenger, na.rm = TRUE),
      avg_r_squared = mean(r_squared, na.rm = TRUE)
    )
})

print(serial_multi_reg_summary)


#### Parallel Multiple Regression with `future_lapply`


In [ ]:
%%R
plan(multisession, workers = max(1L, availableCores() - 1L))
system.time({
  parallel_multi_reg <- future_lapply(data_splits, process_multiple_regression)
  parallel_multi_reg_summary <- do.call(rbind, lapply(parallel_multi_reg, as.data.frame)) %>%
    summarise(
      avg_intercept = mean(intercept, na.rm = TRUE),
      avg_slope_duration = mean(slope_duration, na.rm = TRUE),
      avg_slope_distance = mean(slope_distance, na.rm = TRUE),
      avg_slope_passenger = mean(slope_passenger, na.rm = TRUE),
      avg_r_squared = mean(r_squared, na.rm = TRUE)
    )
})

print(parallel_multi_reg_summary)


### Visualizing Results

Scatter plot of `fare_amount` vs. `trip_duration` with a linear smooth (sampled rows for speed):


In [ ]:
%%R
set.seed(123)
sample_data <- taxi_data %>% slice_sample(n = 1000)

library(ggplot2)
ggplot(sample_data, aes(x = trip_duration, y = fare_amount)) +
  geom_point(alpha = 0.3) +
  geom_smooth(method = "lm", color = "red") +
  labs(x = "Trip Duration (minutes)", y = "Fare Amount ($)", title = "Fare vs. Trip Duration")


## Summary and Conclusion

Parallel data processing is a critical technique for managing large-scale data and computationally intensive tasks. By distributing workloads across multiple processing units, it achieves significant speedups, scalability, and efficiency compared to sequential methods. It relies on effective data partitioning, task coordination, and robust frameworks like Spark or Hadoop. As data volumes grow and computational demands increase, parallel processing remains essential for industries and research fields requiring fast, scalable solutions.

The `{future}` package simplifies parallel processing for statistical analysis of large datasets like the NYC Yellow Taxi data. By replacing `lapply()` with `future_lapply()` and setting `plan(multisession)`, we efficiently computed descriptive statistics, correlations, and regression models. The package's flexibility and automatic handling of globals make it a powerful alternative to `{parallel}`, especially for users seeking a unified, cross-platform solution. For further exploration, consider using `furrr` for `{purrr}`-style workflows or `future.batchtools` for HPC clusters.


## References

1. CRAN: https://cran.r-project.org/package=future
2. GitHub: https://github.com/HenrikBengtsson/future
3. Ecosystem: {future.apply}, {furrr}, {future.batchtools}, {clustermq}
